In [1]:
import langchain

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

True

In [3]:
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")  # Access the GEMINI API key from environment

### Example 1: Simple LLM CALL


In [4]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage,SystemMessage


In [5]:
model=init_chat_model("gemini-3.6-flash", model_provider="google_genai",temperature=0.7)

In [6]:
messages=[
    SystemMessage("You are a helpful AI assistant."),
    HumanMessage("What are the benefits of using LangChain for building AI applications?")
]

In [8]:
## invoke the model
response=model.invoke(messages)

d:\From Old laptop\Visual studio code\AgenticAndGenAI\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [9]:
print(response)

content=[{'type': 'text', 'text': '**LangChain** is an open-source framework designed to simplify the process of building applications powered by Large Language Models (LLMs). Instead of interacting directly with raw LLM APIs, developers use LangChain to construct complex, context-aware, and agentic AI systems.\n\nHere are the primary benefits of using LangChain for AI application development:\n\n---\n\n### 1. Model Agnosticism and Vendor Lock-In Prevention\n* **Unified Interface:** LangChain provides a standardized interface for dozens of different LLMs (OpenAI, Anthropic, Google Gemini, Ollama, Hugging Face, etc.). \n* **Easy Swapping:** You can switch your underlying LLM provider with a single line of code change. This makes it easy to test different models for cost, latency, or performance without rewriting your application logic.\n\n### 2. Built-in Retrieval-Augmented Generation (RAG) Tools\nConnecting LLMs to custom data (PDFs, databases, Notion, web pages) is one of the most com

In [ ]:
###Dynamic Prompt Template

In [16]:
from langchain_core.prompts import ChatPromptTemplate

translation_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that translates following text '{text}' from "
     "{input_language} to {output_language}."),
    ("user", "{text}")
])

In [17]:
prompt=translation_template.invoke({
    "input_language":"English",
    "output_language":"French",
    "text":"Hello, how are you?"
})
prompt

ChatPromptValue(messages=[SystemMessage(content="You are a helpful assistant that translates following text 'Hello, how are you?' from English to French.", additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={})])

In [ ]:
translated_resonse=model.invoke(prompt)
print(translated_resonse.content)

In [ ]:
###Building first Chain

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

def create_story_chain():
    #story generation template
    story_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a creative storyteller. Generate a short story based on the given thereme."),
        ("user", "Theme: {theme}\nMain Character: {character}\nSetting: {setting}")
    ])
    
    #story analysis template
    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a literacy critic. Analyze the following story and provide insights."),
        ("user", "{story}")
    ])
    
    #build chain - Method 1: sequential execution
    story_chain = (
        story_prompt
        | model
        | StrOutputParser()
    )
    
    #create function to pass story to analysis
    def analyze_story(story_text):
        return {"story": story_text}
    
    analysis_chain = (
        story_chain
        | RunnableLambda(analyze_story)
        | analysis_prompt
        | model
        | StrOutputParser()
    )
    return analysis_chain
     